# HPO Launcher — Hyperparameter Optimisation for DARTS-Cards

This notebook runs two HPO strategies over the augment-phase hyperparameters of the architecture found by DARTS:

| Strategy | Script | Sampler |
|---|---|---|
| Random Search (baseline) | `hpo_random_search_baseline.py` | `optuna.samplers.RandomSampler` |
| TPE | `hpo_tpe.py` | `optuna.samplers.TPESampler` (multivariate) |

Both scripts are **resumable**: results are saved to Drive as a CSV after each trial, and the Optuna study is re-seeded from that CSV if the Colab runtime disconnects.

**Run order:** Cells 1–6 (setup) → **Cell 7 (smoke test)** → **Cell 8 (mini HPO test)** → Cell 9 (Random Search) → Cell 10 (TPE) → Cell 11 (results analysis).

> Run Cells 7 and 8 first to confirm everything works before committing to the full run (~10 h total).

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone the repo (or pull latest changes)
import os
repo_path = '/content/darts-cards'
if not os.path.exists(repo_path):
    !git clone https://github.com/javiergarciaduran/darts-cards.git {repo_path}
else:
    !git -C {repo_path} pull
%cd {repo_path}

In [ ]:
# 3. Install dependencies
!pip install -q optuna graphviz wandb tensorboardX numpy
!apt-get install -q graphviz

In [ ]:
# 4. Fix encoding (datasets/__init__.py may be UTF-16 on Windows)
!iconv -f UTF-16 -t UTF-8 /content/darts-cards/datasets/__init__.py > temp.py && mv temp.py /content/darts-cards/datasets/__init__.py

In [ ]:
# 5. Symlink dataset from Drive and verify
!mkdir -p ./data
!ln -sfn /content/drive/MyDrive/cards ./data/cards

from datasets.cards import get_cards
tr, nc = get_cards('./data/cards', split='train')
va, _  = get_cards('./data/cards', split='val')
print(f'train={len(tr)}  val={len(va)}  classes={nc}')

In [ ]:
# 5b. Copy dataset to local disk to avoid slow Drive I/O during training.
# hpo_common.py hardcodes --data_path ./data/cards, so we re-point the symlink
# to /tmp/cards after copying. All scripts transparently use local disk.
# Runs once per session (~2–5 min depending on dataset size).
import os, shutil

LOCAL_DATA_PATH = '/tmp/cards'

if os.path.exists(LOCAL_DATA_PATH):
    print(f'Dataset already on local disk at {LOCAL_DATA_PATH} — skipping copy.')
else:
    print('Copying dataset from Drive to local disk...')
    shutil.copytree('/content/drive/MyDrive/cards', LOCAL_DATA_PATH)
    print('Copy done.')

# Re-point ./data/cards → /tmp/cards (was → Drive)
!ln -sfn {LOCAL_DATA_PATH} ./data/cards
print('Symlink updated: ./data/cards →', os.path.realpath('./data/cards'))

# Verify
from datasets.cards import get_cards
tr, nc = get_cards('./data/cards', split='train')
va, _  = get_cards('./data/cards', split='val')
print(f'Local copy verified: train={len(tr)}  val={len(va)}  classes={nc}')

In [ ]:
# 6. Create output directories on Drive
!mkdir -p /content/drive/MyDrive/darts_experiments/hpo_random_search
!mkdir -p /content/drive/MyDrive/darts_experiments/hpo_tpe
!mkdir -p /content/drive/MyDrive/darts_experiments/hpo_logs
print('Output directories ready.')

---
## 🧪 Test cells — run these before the real HPO

**Cell 7** checks that `augment.py` itself runs correctly (3 epochs, ~3 min).  
**Cell 8** checks the full HPO loop end-to-end (1 trial × 3 epochs per strategy, ~6 min).  
Both use temp paths — nothing is written to Drive.

If both cells finish without errors you are ready to run the real HPO.

In [ ]:
# 7. Smoke test — verify augment.py runs for 3 epochs with the known-good config
# Expected output: live training logs, a 'Final best Prec@1' line, PASSED message.
# Expected time: ~1 min on L4 (with data on local disk).
import subprocess, sys
from hpo_common import GENOTYPE

proc = subprocess.Popen(
    [sys.executable, '-u', 'augment.py',
     '--name', 'smoke_test',
     '--dataset', 'cards',
     '--data_path', LOCAL_DATA_PATH,   # /tmp/cards — local disk, set in Cell 5b
     '--path', '/tmp/smoke_test',
     '--batch_size', '96',
     '--init_channels', '24',
     '--layers', '14',
     '--epochs', '3',
     '--lr', '0.025',
     '--weight_decay', '3e-4',
     '--drop_path_prob', '0.2',
     '--aux_weight', '0.4',
     '--cutout_length', '8',
     '--momentum', '0.9',
     '--grad_clip', '5.0',
     '--print_freq', '50',
     '--workers', '2',
     '--gpus', '0',
     '--seed', '42',
     '--genotype', GENOTYPE],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print('\n✅ Smoke test PASSED — augment.py ran 3 epochs without errors.')
else:
    print(f'\n❌ Smoke test FAILED — exit code {proc.returncode}. Fix the error above before continuing.')

In [ ]:
# 8. Mini HPO test — 1 trial × 3 epochs for each HPO strategy
# Temporarily patches the epoch count in hpo_common.py (50 → 3), runs both
# scripts, then restores the original file. Nothing written to Drive.
# Expected time: ~6 min on L4.
import subprocess, sys, csv

def run_and_stream(cmd):
    """Run cmd as a subprocess, stream stdout live to the cell, return exit code."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

# Patch
!sed -i 's/"--epochs", "50"/"--epochs", "3"/' hpo_common.py
print('Patched hpo_common.py: epochs 50 → 3\n')

errors = []

print('--- Running Random Search (1 trial, 3 epochs) ---')
rc1 = run_and_stream([sys.executable, 'hpo_random_search_baseline.py',
                      '--n_trials', '1',
                      '--hpo_output_dir', '/tmp/hpo_test_rs',
                      '--csv_path', '/tmp/hpo_test_rs_results.csv'])
if rc1 != 0:
    errors.append(f'Random Search exited with code {rc1}')

print('\n--- Running TPE (1 trial, 3 epochs) ---')
rc2 = run_and_stream([sys.executable, 'hpo_tpe.py',
                      '--n_trials', '1',
                      '--hpo_output_dir', '/tmp/hpo_test_tpe',
                      '--csv_path', '/tmp/hpo_test_tpe_results.csv'])
if rc2 != 0:
    errors.append(f'TPE exited with code {rc2}')

# Restore (always, even on error)
!git checkout hpo_common.py
print('\nRestored hpo_common.py')

# Verify restore
with open('hpo_common.py') as f:
    content = f.read()
assert '"--epochs", "50"' in content, 'RESTORE FAILED — hpo_common.py still patched!'

# Inspect CSV outputs
for label, path in [('Random Search', '/tmp/hpo_test_rs_results.csv'),
                    ('TPE',           '/tmp/hpo_test_tpe_results.csv')]:
    try:
        with open(path) as f:
            rows = list(csv.DictReader(f))
        print(f'{label} CSV: {len(rows)} row(s) — value={rows[0]["value"] if rows else "n/a"}')
    except FileNotFoundError:
        errors.append(f'{label} CSV not found at {path}')

if errors:
    print('\n❌ Mini HPO test FAILED:')
    for e in errors:
        print(' •', e)
else:
    print('\n✅ Mini HPO test PASSED — both strategies completed 1 trial without errors.')
    print('Ready to run the full HPO (Cells 9 and 10).')

---
## Random Search (baseline)

Runs `--n_trials` augment.py trials with hyperparameters sampled uniformly at random.
Each trial trains for **50 epochs** (fast proxy for final accuracy).

Results are appended to the CSV on Drive after every trial — safe to interrupt and resume.

In [ ]:
!python hpo_random_search_baseline.py \
    --n_trials 20 \
    --hpo_seed 42 \
    --data_path /tmp/cards \
    --hpo_output_dir /content/drive/MyDrive/darts_experiments/hpo_random_search \
    --csv_path /content/drive/MyDrive/darts_experiments/hpo_results_rs.csv \
    2>&1 | tee /content/drive/MyDrive/darts_experiments/hpo_logs/random_search.log

## TPE Search

Runs `--n_trials` augment.py trials guided by a multivariate TPE sampler.
The first trial always uses the known-good baseline config from `cards_augment_v1`
(lr=0.025, wd=3e-4, drop_path=0.2, aux=0.4, cutout=8) to warm-start the surrogate model.

Results are appended to the CSV on Drive after every trial — safe to interrupt and resume.

In [ ]:
!python hpo_tpe.py \
    --n_trials 20 \
    --hpo_seed 42 \
    --data_path /tmp/cards \
    --hpo_output_dir /content/drive/MyDrive/darts_experiments/hpo_tpe \
    --csv_path /content/drive/MyDrive/darts_experiments/hpo_tpe_results.csv \
    2>&1 | tee /content/drive/MyDrive/darts_experiments/hpo_logs/tpe.log

## Results Analysis

Load both CSVs from Drive, print the best trial for each strategy, and plot the optimisation history.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math

RS_CSV  = '/content/drive/MyDrive/darts_experiments/hpo_results_rs.csv'
TPE_CSV = '/content/drive/MyDrive/darts_experiments/hpo_tpe_results.csv'
BASELINE = {'lr': 0.025, 'weight_decay': 3e-4, 'drop_path_prob': 0.2,
            'aux_weight': 0.4, 'cutout_length': 8}

rs  = pd.read_csv(RS_CSV)
tpe = pd.read_csv(TPE_CSV)

# Filter out failed trials (value == -1)
rs_ok  = rs[rs['value'] != -1.0].copy()
tpe_ok = tpe[tpe['value'] != -1.0].copy()

print('=== Random Search ===')
print(f'Completed trials : {len(rs_ok)} / {len(rs)}')
best_rs = rs_ok.loc[rs_ok['value'].idxmax()]
print(f'Best Prec@1      : {best_rs["value"]*100:.2f}%')
print(best_rs.to_string())

print('\n=== TPE ===')
print(f'Completed trials : {len(tpe_ok)} / {len(tpe)}')
best_tpe = tpe_ok.loc[tpe_ok['value'].idxmax()]
print(f'Best Prec@1      : {best_tpe["value"]*100:.2f}%')
print(best_tpe.to_string())

# Running best (cumulative max) for each strategy
rs_ok['best_so_far']  = rs_ok['value'].cummax()
tpe_ok['best_so_far'] = tpe_ok['value'].cummax()

# Plot 1: optimisation history (individual trials + running best)
fig1, ax = plt.subplots(figsize=(8, 5))
ax.scatter(rs_ok['trial_number'],  rs_ok['value'],  alpha=0.6, label='RS trials',  color='steelblue')
ax.scatter(tpe_ok['trial_number'], tpe_ok['value'], alpha=0.6, label='TPE trials', color='darkorange')
ax.plot(rs_ok['trial_number'],  rs_ok['best_so_far'],  '--', color='steelblue',  label='RS best so far')
ax.plot(tpe_ok['trial_number'], tpe_ok['best_so_far'], '--', color='darkorange', label='TPE best so far')
ax.set_xlabel('Trial number')
ax.set_ylabel('Validation Prec@1')
ax.set_title('Optimisation history')
ax.legend()
ax.grid(True, alpha=0.3)
fig1.tight_layout()
plot_path1 = '/content/drive/MyDrive/darts_experiments/hpo_logs/hpo_history.png'
fig1.savefig(plot_path1, dpi=150)
print(f'\nSaved plot → {plot_path1}')
plt.show()

# Plot 2: best hyperparameter values side-by-side
# cutout_length is divided by 16 so it doesn't dwarf the other parameters.
params = ['lr', 'weight_decay', 'drop_path_prob', 'aux_weight', 'cutout_length']
labels = ['lr', 'weight_decay', 'drop_path_prob', 'aux_weight', 'cutout_length\n(÷ 16)']
x = np.arange(len(params))
bar_w = 0.25

def plot_val(cfg, key):
    v = cfg[key]
    return v / 16 if key == 'cutout_length' else v

def fmt_val(cfg, key):
    v = cfg[key]
    if key == 'lr':            return f'{v:.4f}'
    if key == 'weight_decay':  return f'{v:.0e}'
    if key == 'cutout_length': return f'{v/16: .2f}'
    if key == 'drop_path_prob':return f'{v: .2f}'
    return f'{v:.3f}'

configs = [
    (BASELINE,           'Baseline',  'gray'),
    (best_rs.to_dict(),  'RS best',   'steelblue'),
    (best_tpe.to_dict(), 'TPE best',  'darkorange'),
]
offsets = [-bar_w, 0, bar_w]

fig2, ax = plt.subplots(figsize=(8, 5))
for (cfg, label, color), offset in zip(configs, offsets):
    bars = ax.bar(
        x + offset,
        [plot_val(cfg, p) for p in params],
        bar_w, label=label, color=color, alpha=0.8,
    )
    for bar, p in zip(bars, params):
        ax.annotate(
            fmt_val(cfg, p),
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords='offset points',
            ha='center', va='bottom', fontsize=7, rotation=90,
        )

ax.set_xticks(x)
ax.set_ylim(bottom=0, top=0.6)
ax.set_xticklabels(labels, rotation=25, ha='right')
ax.set_title('Best hyperparameters')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
fig2.tight_layout()
plot_path2 = '/content/drive/MyDrive/darts_experiments/hpo_logs/hpo_hyperparams.png'
fig2.savefig(plot_path2, dpi=150)
print(f'Saved plot → {plot_path2}')
plt.show()

---
## 200-Epoch Final Training with HPO-Best Hyperparameters

The HPO searches (Random Search and TPE, 20 trials each at 50 epochs) identified the
best hyperparameter configurations for training the DARTS-derived architecture.
Here we re-train with those best configs for the full **200 epochs** so the result is
directly comparable to the NAS-only baseline (also trained for 200 epochs).

| Method | Best val Prec@1 (50 ep) | lr | weight_decay | drop_path_prob | aux_weight | cutout_length |
|--------|------------------------|----|--------------|----------------|------------|---------------|
| Random Search | 84.91% | 0.03288 | 7.26e-4 | 0.0524 | 0.404 | 8 |
| TPE | 83.40% | 0.03490 | 2.50e-4 | 0.0724 | 0.340 | 8 |

Results are written to Google Drive under `darts_experiments/hpo_best_rs/` and
`darts_experiments/hpo_best_tpe/`. Each run takes roughly **4× longer** than the
50-epoch HPO proxy (~2 h per run on a T4 GPU).

In [ ]:
import subprocess, sys
import pandas as pd
from hpo_common import GENOTYPE

PARAM_COLS = ["lr", "weight_decay", "drop_path_prob", "aux_weight", "cutout_length"]

def best_params_from_csv(csv_path):
    """Return the hyperparameter dict of the highest-value trial in the CSV."""
    df = pd.read_csv(csv_path)
    df = df[df["value"] != -1.0]
    best_row = df.loc[df["value"].idxmax()]
    params = {col: best_row[col] for col in PARAM_COLS}
    params["cutout_length"] = int(params["cutout_length"])
    print(f"  Best trial #{int(best_row['trial_number'])}  val={best_row['value']*100:.2f}%")
    for k, v in params.items():
        print(f"    {k}: {v}")
    return params

BASE_PATH = "/content/drive/MyDrive/darts_experiments"
EPOCHS = 200
runs = [
    ("rs",  f"{BASE_PATH}/hpo_results_rs.csv"),
    ("tpe", f"{BASE_PATH}/hpo_tpe_results.csv"),
]

def run_augment(name, params):
    out_path = f"{BASE_PATH}/hpo_best_{name}"
    cmd = [
        sys.executable, "-u", "augment.py",
        "--name",           f"hpo_best_{name}",
        "--dataset",        "cards",
        "--data_path",      LOCAL_DATA_PATH,
        "--path",           out_path,
        "--epochs",         str(EPOCHS),
        "--lr",             str(params["lr"]),
        "--weight_decay",   str(params["weight_decay"]),
        "--drop_path_prob", str(params["drop_path_prob"]),
        "--aux_weight",     str(params["aux_weight"]),
        "--cutout_length",  str(params["cutout_length"]),
        "--genotype",       GENOTYPE,
    ]
    print(f"\n{'='*60}")
    print(f"Starting {name.upper()} best-params run — {EPOCHS} epochs")
    print(f"Output path : {out_path}")
    print(f"{'='*60}\n")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode == 0:
        print(f"\n✅ {name.upper()} 200-epoch run DONE — checkpoints saved to {out_path}")
    else:
        print(f"\n❌ {name.upper()} run FAILED (exit code {proc.returncode})")


for name, csv_path in runs:
    print(f"\n--- {name.upper()} best params (from {csv_path}) ---")
    params = best_params_from_csv(csv_path)
    run_augment(name, params)

---
### Test set evaluation (run only once)

In [ ]:
import torch
from torch.utils.data import DataLoader
from datasets.cards import get_cards
from models.augment_cnn import AugmentCNN
from genotypes import CARDS_V1

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_PATH = "/content/drive/MyDrive/darts_experiments"

CHECKPOINTS = {
    "HPO – RS":  f"{BASE_PATH}/hpo_best_rs/hpo_best_rs/best.pth.tar",
    "HPO – TPE": f"{BASE_PATH}/hpo_best_tpe/hpo_best_tpe/best.pth.tar",
}

def load_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    if isinstance(ckpt, dict):
        cfg = ckpt['config']
        model = AugmentCNN(
            input_size=32, C_in=3, C=cfg.init_channels,
            n_classes=cfg.n_classes, n_layers=cfg.layers,
            auxiliary=False, genotype=CARDS_V1,
        ).to(DEVICE)
        model.load_state_dict(ckpt['state_dict'])
    else:
        model = ckpt.module if hasattr(ckpt, 'module') else ckpt
        model = model.to(DEVICE)
    return model

test_data, _ = get_cards(LOCAL_DATA_PATH, split='test')
test_loader  = DataLoader(test_data, batch_size=128, shuffle=False,
                          num_workers=2, pin_memory=True)

test_results = {}   # populated here, consumed by the barplot cell below

for name, ckpt_path in CHECKPOINTS.items():
    print(f"\n{'─'*50}")
    print(f"Evaluating: {name}")
    model = load_model(ckpt_path)
    model.eval()

    top1_correct = top5_correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits, _ = model(images)
            top1_correct += (logits.argmax(dim=1) == labels).sum().item()
            top5_idx     = logits.topk(5, dim=1).indices
            top5_correct += sum(
                labels[i].item() in top5_idx[i].tolist()
                for i in range(labels.size(0))
            )
            total += labels.size(0)

    top1 = 100 * top1_correct / total
    top5 = 100 * top5_correct / total
    test_results[name] = {"top1": top1, "top5": top5}
    print(f"  Top-1: {top1:.2f}%   Top-5: {top5:.2f}%   (n={total})")

print(f"\n{'='*50}")
print(f"{'Model':<22} {'Top-1':>7} {'Top-5':>7}")
print(f"{'─'*36}")
for name, acc in test_results.items():
    print(f"{name:<22} {acc['top1']:>6.2f}% {acc['top5']:>6.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Fill in the NAS baseline numbers from darts_launcher.ipynb Cell 4 ──

all_results = {
    "NAS": {"top1": 82.64, "top5": 90.19},
    "HPO – RS": {"top1": 84.50, "top5": 91.30},
    "HPO – TPE": {"top1": 83.40, "top5": 93.60},
    "EfficientNet B4 + NAS": {"top1": 72.83, "top5": 94.34},
}

names  = list(all_results.keys())
top1   = [all_results[n]["top1"] for n in names]
top5   = [all_results[n]["top5"] for n in names]

x      = np.arange(len(names))
bar_w  = 0.35
colors = ["#4C72B0", "#DD8452", "#55A868", "#8172B2"]

fig, ax = plt.subplots(figsize=(9, 5))

bars1 = ax.bar(x - bar_w / 2, top1, bar_w, label="Top-1",
               color=colors, alpha=0.9)
bars5 = ax.bar(x + bar_w / 2, top5, bar_w, label="Top-5",
               color=colors, alpha=0.5, hatch="//")

for bar in [*bars1, *bars5]:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.4,
        f"{bar.get_height():.1f}%",
        ha="center", va="bottom", fontsize=9,
    )

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Test Accuracy — NAS Baseline vs HPO-Tuned Networks")
ax.set_ylim(0, max(top5) + 8)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plot_path = f"{BASE_PATH}/hpo_logs/test_accuracy_comparison.png"
plt.savefig(plot_path, dpi=150)
print(f"Saved → {plot_path}")
plt.show()